# Polarimetric signatures on geocoded ALOS-1 data

This notebook checks `polarimetric_signature` on the geocoded San Francisco ALOS-1 T3 product. It displays a Pauli RGB image, selects three pixels by integer row and column position, and plots their signatures. No correspondence with pixels in the slant-range product is assumed because no mapping LUT is available.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

from polsarpro.io import open_netcdf_beam
from polsarpro.polarisation import (
    plot_polarimetric_signature,
    polarimetric_signature,
)
from polsarpro.util import pauli_rgb

input_file = Path(
    "/data/psp/test_files/SAN_FRANCISCO_ALOS1_geocoded_T3_7_look_az.nc"
)

## Open the geocoded T3 product

In [ ]:
T3 = open_netcdf_beam(input_file)
T3

## Display the selected pixels

`row` and `col` are zero-based array positions along the `lat` and `lon` dimensions.

In [ ]:
points = {
    "Point A": (1200, 800),
    "Point B": (1450, 1350),
    "Point C": (2600, 1100),
}

with ProgressBar():
    pauli = pauli_rgb(T3).compute()

figure, axis = plt.subplots(figsize=(7, 7))
axis.imshow(pauli.transpose("lat", "lon", "band").values, origin="upper")
for name, (row, col) in points.items():
    axis.scatter(col, row, s=45, edgecolor="white", label=name)
axis.set(title="Selected geocoded pixels", xlabel="Column", ylabel="Row")
axis.legend()
plt.show()

## Compute the signatures

In [ ]:
signatures = {}
with ProgressBar():
    for name, (row, col) in points.items():
        signatures[name] = polarimetric_signature(T3, row=row, col=col)

## Plot the signatures

In [ ]:
for name, signature in signatures.items():
    figure, axes = plot_polarimetric_signature(signature)
    figure.suptitle(name)
plt.show()